# profile_runs

> Phase 13 (Track A, A3): runs of the hidden-profile vote. `profileCell` builds one episode's cell, `runProfile` plays cells in batches through `claudePolicy` (each round's requests at once, then every vote), and returns one row per episode keyed with `cellRow`, with a turn log per cell.

In [ ]:
#| default_exp profile_runs

In [ ]:
#| hide
from nbdev.showdoc import *

**Cells.** A cell is one episode, built by `profileCell(study, arm, variant, seed, rounds=3, N=4, model=HAIKU, effort='low', maxTokens=8000, **task)`: `dict(study, factor='arm', level='<arm>|<variant>', seed, runner='llm', cfg)`, where `cfg` is an `LLMConfig` dict with `task='profile'`, `channel='blackboard'`, `N`, `turns` = discussion rounds, `model`, `effort`, `maxTokens`, and `world` = `dict(arm, variant, rule, table, solo, [hint], [domain, shared, unique])`. `arm` is a key of `ARMS` (`baseline`, `factsFirst`, `table`, `dissent`) or `solo` (the one-agent ceiling: one request per episode); `runProfile` reads the settings from `rule`, `table`, `hint` and `solo`, so a grid can also spell them out itself. The seed is the episode seed (shared by the arms and the solo ceiling, so they pair). A cell may carry `client` and `logDir` keys (`withClient`); `runProfile` ignores them and uses the client and directory it is given.

**Rows.** `cellRow` columns (study, factor, level, seed, runner, budget, the config, `world.*`), then `episode` (0), `domain`, `success` (strict majority for the hidden-best; solo: its vote), `votesA`, `factsPosted` (distinct facts on the board), `factsHeld` (facts in the task), `uniquePosted` (private facts posted / private facts held), `opinions`, `invalid` (invalid posts and votes), `calls`, `cost`, `latency` (sequential request stages: rounds + the vote, solo 1; wall-clock time is in the client's timing records), refusals, max-token stops and token totals. Solo rows have NaN for the board columns.

**Turn logs.** One JSONL file per cell in `logDir`, named by the row key: per request `agent`, `round` (the vote is round `rounds + 1`), `kind` (`post`, `vote` or `solo`), `system`, `prompt`, `reply`, `parsed`, `valid`, `stop`, `refusal`, `usage`, `cost`, with the row `key` and `episode`.

In [ ]:
#| export
import json
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

from explRL.bench_grid import rowKey
from explRL.clue_policy import claudePolicy, historyBlocks, userText
from explRL.hidden_profile import POST_SCHEMA, ProfileEnv, parseVote, profileTask, soloCeiling, voteSchema
from explRL.llm_budget import estimateCost
from explRL.llm_grid import HAIKU
from explRL.llm_runs import LLMConfig, cellRow

ARMS = dict(baseline={}, factsFirst=dict(rule='factsFirst'), table=dict(table=True), dissent=dict(hint='dissent'))
SOLO_SYSTEM = 'You are making a decision on your own. Reply with the JSON object the schema asks for.'
NAN = float('nan')


def profileCell(study, arm, variant, seed, rounds=3, N=4, model=HAIKU, effort='low', maxTokens=8000, **task):
    """One episode of the vote: arm in ARMS or 'solo' (the one-agent ceiling); `task`: profileTask's domain, shared,
    unique. The arm's settings are spelled out in cfg.world, which runProfile reads."""
    assert arm in ARMS or arm == 'solo', arm
    world = {**dict(arm=arm, variant=variant, rule='free', table=False, solo=arm == 'solo'), **ARMS.get(arm, {}), **task}
    cfg = LLMConfig(task='profile', channel='blackboard', N=N, turns=rounds, model=model, effort=effort,
                    maxTokens=maxTokens, memory='notes', world=world)
    return dict(study=study, factor='arm', level=f'{arm}|{variant}', seed=seed, runner='llm', cfg=cfg.asDict())


def _task(cfg, seed):
    w = cfg.world
    kw = {k: tuple(w[k]) if k == 'shared' else w[k] for k in ('domain', 'shared', 'unique') if k in w}
    return profileTask(seed, w['variant'], N=cfg.N, **kw)


def _estimate(cfg):
    return estimateCost(cfg.model, 'profile', 1, 1) if cfg.world.get('solo') else \
        estimateCost(cfg.model, 'profile', cfg.N, cfg.turns + 1)


def _ask(client, groups, workers):
    "groups: [(schema, [(system, blocks)])] -> replies per group, each group through claudePolicy, groups concurrently."
    if not groups:
        return []
    pol = lambda g: claudePolicy(client, workers, g[0], 'profile')(g[1])
    with ThreadPoolExecutor(max(1, min(workers, len(groups)))) as ex:
        return list(ex.map(pol, groups))


def _rec(agent, rnd, kind, prompt, rep, valid):
    return dict(agent=agent, round=rnd, kind=kind, system=prompt[0], prompt=userText(prompt[1]), reply=rep.text,
                parsed=rep.parsed, valid=bool(valid), stop=rep.stopReason, refusal=rep.refusal, usage=rep.usage,
                cost=rep.cost)


def _batch(cells, client, workers):
    "Play a batch of episodes: each round's requests at once, then every vote (and solo request) at once."
    eps = []
    for c in cells:
        cfg = LLMConfig.fromDict(c['cfg'])
        task = _task(cfg, c['seed'])
        w = cfg.world
        env = None if w.get('solo') else ProfileEnv(task, cfg.turns, w.get('rule', 'free'), w.get('table', False), w.get('hint'))
        eps.append(dict(cell=c, cfg=cfg, task=task, env=env, notes=[''] * cfg.N, log=[], votes=[]))
    teams = [e for e in eps if e['env'] is not None]
    for r in range(1, max([e['cfg'].turns for e in teams], default=0) + 1):
        items = [(e, i, (e['env'].rules(), historyBlocks('notes', e['notes'][i]) + [{'type': 'text', 'text': 'Now:\n' + e['env'].observe(i)}]))
                 for e in teams if r <= e['cfg'].turns for i in range(e['cfg'].N)]
        for (e, i, p), rep in zip(items, _ask(client, [(POST_SCHEMA, [p for _, _, p in items])], workers)[0]):
            e['log'].append(_rec(i, r, 'post', p, rep, e['env'].post(i, rep.parsed)))
            if isinstance(rep.parsed, dict) and isinstance(rep.parsed.get('notes'), str):
                e['notes'][i] = rep.parsed['notes'][:1000]
        for e in teams:
            if r <= e['cfg'].turns:
                e['env'].endRound()
    groups, owners = [], []
    for e in eps:
        env, schema = e['env'], voteSchema(e['task']['names'])
        if env is None:
            ps = [(SOLO_SYSTEM, [{'type': 'text', 'text': soloCeiling(e['task'])}])]
        else:
            ps = [(env.rules(), historyBlocks('notes', e['notes'][i]) + [{'type': 'text', 'text': 'Now:\n' + env.voteText(i)}])
                  for i in range(e['cfg'].N)]
        groups.append((schema, ps))
        owners.append(e)
    for e, (_, ps), reps in zip(owners, groups, _ask(client, groups, workers)):
        for i, (p, rep) in enumerate(zip(ps, reps)):
            v = e['env'].vote(i, rep.parsed) if e['env'] else parseVote(e['task']['names'], rep.parsed)
            e['votes'].append(v)
            e['log'].append(_rec(i, e['cfg'].turns + 1 if e['env'] else 1, 'vote' if e['env'] else 'solo', p, rep, v is not None))
    return eps


def _row(e):
    "One row: the task, the outcome, the board's statistics, the replies' totals."
    task, env, votes, log = e['task'], e['env'], e['votes'], e['log']
    best = task['best']
    if env is None:
        st = dict(success=int(votes[0] == best), factsPosted=NAN, factsHeld=len(task['facts']), uniquePosted=NAN,
                  opinions=NAN, invalid=int(votes[0] is None))
    else:
        st = dict(success=int(env.success(votes)), **env.stats())
    u = {k: sum(x['usage'].get(k, 0) for x in log) for k in
         ('input_tokens', 'cache_creation_input_tokens', 'cache_read_input_tokens', 'output_tokens')}
    return dict(episode=0, domain=task['domain'], **st, votesA=sum(v == best for v in votes),
                calls=len(log), cost=float(sum(x['cost'] for x in log)), latency=1 if env is None else e['cfg'].turns + 1,
                refusals=sum(x['refusal'] is not None for x in log), maxTokenStops=sum(x['stop'] == 'max_tokens' for x in log),
                inputTokens=u['input_tokens'], cacheWriteTokens=u['cache_creation_input_tokens'],
                cacheReadTokens=u['cache_read_input_tokens'], outputTokens=u['output_tokens'])


def runProfile(cells, client, logDir=None, batch=100, workers=8):
    """Run vote cells (profileCell dicts; one episode each, seed = episode seed) in batches through claudePolicy; rows
    keyed with cellRow, sorted by key; one turn log per cell in `logDir`."""
    client.reserve(sum(_estimate(LLMConfig.fromDict(c['cfg'])) for c in cells))
    rows = []
    for s in range(0, len(cells), batch):
        for e in _batch(cells[s:s + batch], client, workers):
            row = cellRow(e['cell'], e['cfg'], _row(e))
            rows.append(row)
            if logDir:
                p = Path(logDir) / (rowKey(row).replace('|', '__') + '.jsonl')
                p.parent.mkdir(parents=True, exist_ok=True)
                p.write_text(''.join(json.dumps(dict(key=rowKey(row), episode=0, **x), sort_keys=True) + '\n' for x in e['log']))
    return sorted(rows, key=rowKey)

## Tests

In [ ]:
import json, tempfile
from pathlib import Path
from explRL.profile_runs import *
from explRL.bench_grid import readCSV, rowKey, writeCSV
from explRL.llm_budget import estimateCost
from explRL.llm_client import LLMClient, ResponseCache
from explRL.llm_fake import FakeClient, fakeReply
from explRL.llm_grid import HAIKU

# cells: every arm and the solo ceiling, both variants, three episode seeds (shared by the arms)
cells = [profileCell('profileA', arm, v, s) for arm in (*ARMS, 'solo') for v in ('classic', 'lone') for s in (5000, 5001, 5002)]
c = cells[0]
assert c['level'] == 'baseline|classic' and c['factor'] == 'arm' and c['cfg']['task'] == 'profile' and c['cfg']['model'] == HAIKU
assert (c['cfg']['effort'], c['cfg']['maxTokens'], c['cfg']['turns'], c['cfg']['N']) == ('low', 8000, 3, 4)
assert profileCell('x', 'table', 'lone', 1)['cfg']['world'] == dict(arm='table', variant='lone', rule='free', table=True, solo=False)
assert profileCell('x', 'dissent', 'classic', 1, shared=(1, 1, 2))['cfg']['world']['hint'] == 'dissent'

tmp = Path(tempfile.mkdtemp())
client = LLMClient(HAIKU, client=FakeClient(fakeReply), cache=ResponseCache(tmp / 'cache'))
rows = runProfile(cells, client, tmp / 'logs')
team = [r for r in rows if not r['level'].startswith('solo')]
solo = [r for r in rows if r['level'].startswith('solo')]
assert abs(client.pending - (len(team) * estimateCost(HAIKU, 'profile', 4, 4) + len(solo) * estimateCost(HAIKU, 'profile', 1, 1))) < 1e-12
assert len(rows) == len(cells) and [rowKey(r) for r in rows] == sorted(rowKey(r) for r in rows)
assert {r['level'] for r in rows} == {f'{a}|{v}' for a in (*ARMS, 'solo') for v in ('classic', 'lone')}
assert all(r['calls'] == 16 and r['latency'] == 4 and 0 <= r['votesA'] <= 4 for r in team)
assert all(r['success'] == int(r['votesA'] >= 3) and r['factsHeld'] == (14 if 'lone' in r['level'] else 13) for r in team)
assert any(r['factsPosted'] > 0 for r in team) and any(r['opinions'] > 0 for r in team) and all(r['cost'] > 0 for r in rows)
assert all(r['calls'] == 1 and r['votesA'] == r['success'] and r['factsPosted'] != r['factsPosted'] for r in solo)
assert client.calls == 16 * len(team) + len(solo) and {r['domain'] for r in rows} <= {'hiring', 'investment', 'property'}
# factsFirst: no opinion is posted in round 1 (the fake client's round-1 opinions are rejected as invalid)
ff = [r for r in rows if r['level'].startswith('factsFirst')]
assert all(r['invalid'] == 0 for r in rows if r not in ff) and sum(r['invalid'] for r in ff) > 0
# turn logs: one file per cell, one record per request
logs = sorted((tmp / 'logs').glob('*.jsonl'))
assert len(logs) == len(cells)
recs = [json.loads(x) for x in logs[0].read_text().splitlines()]
r0 = next(r for r in rows if rowKey(r) == recs[0]['key'])
assert len(recs) == r0['calls'] and {'agent', 'round', 'kind', 'system', 'prompt', 'reply', 'parsed', 'valid'} <= set(recs[0])
assert [x['round'] for x in recs if x['kind'] == 'vote'] == [4] * 4 and recs[0]['prompt'].startswith('Now:\nRound 1 of 3.')
assert recs[4]['prompt'].startswith('Your notes so far: ') and sum(not x['valid'] for x in recs) == r0['invalid']
# a cached rerun (a fresh client, other batches and order) is identical and makes no calls; CSV and logs byte-identical
again = LLMClient(HAIKU, client=FakeClient(fakeReply), cache=ResponseCache(tmp / 'cache'))
rows2 = runProfile(cells[::-1], again, tmp / 'logs2', batch=5)
assert rows2 == rows and again.calls == 0 and again.hits == client.calls
assert writeCSV(rows, tmp / 'a.csv').read_bytes() == writeCSV(rows2, tmp / 'b.csv').read_bytes()
assert all(a.read_bytes() == (tmp / 'logs2' / a.name).read_bytes() for a in logs)
assert len(readCSV(tmp / 'a.csv')) == len(rows)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()